# 02 — U-Net Training & Evaluation

A U-Net (encoder–decoder with skip connections, base=64, ≈31M parameters) trained
for 7-class semantic segmentation from the 2-channel SAR input.

**Training setup & reasoning**
- **Loss:** CrossEntropy + Dice (Dice helps class overlap under imbalance).
- **Class weighting:** inverse-frequency, so rare classes aren't ignored.
- **Optimizer:** Adam (lr=1e-3); **scheduler:** ReduceLROnPlateau on val mIoU.
- **Early stopping:** patience 12 on val mIoU; best checkpoint saved.
- **Augmentation:** flips + 90° rotations (SAR-safe — no photometric distortion).
- **Variants:** VV+VH (main) vs **VV-only** (to show the value of dual-polarization).

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
CLASSES = ["Water","Trees","Grassland","Cropland","Built-up","Bare/Sparse","Wetland"]; N=7
device = "cuda" if torch.cuda.is_available() else "cpu"

class DoubleConv(nn.Module):
    def __init__(s,cin,cout):
        super().__init__()
        s.net=nn.Sequential(nn.Conv2d(cin,cout,3,padding=1),nn.BatchNorm2d(cout),nn.ReLU(True),
                            nn.Conv2d(cout,cout,3,padding=1),nn.BatchNorm2d(cout),nn.ReLU(True))
    def forward(s,x): return s.net(x)

class UNet(nn.Module):
    def __init__(s,in_ch=2,n_classes=7,base=64):
        super().__init__()
        s.d1=DoubleConv(in_ch,base); s.d2=DoubleConv(base,base*2); s.d3=DoubleConv(base*2,base*4); s.d4=DoubleConv(base*4,base*8)
        s.pool=nn.MaxPool2d(2); s.bott=DoubleConv(base*8,base*16)
        s.up4=nn.ConvTranspose2d(base*16,base*8,2,2); s.u4=DoubleConv(base*16,base*8)
        s.up3=nn.ConvTranspose2d(base*8,base*4,2,2);  s.u3=DoubleConv(base*8,base*4)
        s.up2=nn.ConvTranspose2d(base*4,base*2,2,2);  s.u2=DoubleConv(base*4,base*2)
        s.up1=nn.ConvTranspose2d(base*2,base,2,2);    s.u1=DoubleConv(base*2,base)
        s.out=nn.Conv2d(base,n_classes,1)
    def forward(s,x):
        c1=s.d1(x); c2=s.d2(s.pool(c1)); c3=s.d3(s.pool(c2)); c4=s.d4(s.pool(c3)); b=s.bott(s.pool(c4))
        x=s.u4(torch.cat([s.up4(b),c4],1)); x=s.u3(torch.cat([s.up3(x),c3],1))
        x=s.u2(torch.cat([s.up2(x),c2],1)); x=s.u1(torch.cat([s.up1(x),c1],1)); return s.out(x)

In [ ]:
def dice_loss(logits,target,n=N,eps=1e-6):
    p=F.softmax(logits,1); t=F.one_hot(target,n).permute(0,3,1,2).float()
    inter=(p*t).sum((0,2,3)); union=p.sum((0,2,3))+t.sum((0,2,3))
    return 1-((2*inter+eps)/(union+eps)).mean()

def class_weights(Y,n=N):
    c=np.bincount(Y.ravel(),minlength=n).astype(float)
    return torch.tensor(c.sum()/(n*np.maximum(c,1)),dtype=torch.float32)

# Full loop (Adam + ReduceLROnPlateau + early stopping + best-checkpoint) is in
# train.py, run on the GPU as:  nohup python3 -u train.py > train.log 2>&1 &

## Executed results (held-out test regions)

**Main model — U-Net VV+VH**  ·  best val mIoU 0.545  →  **Test OA 0.757 · mIoU 0.347**

| Class | IoU | F1 |
|---|---|---|
| Water | 0.082 | 0.152 |
| Trees | 0.557 | 0.715 |
| Grassland | 0.219 | 0.359 |
| Cropland | **0.852** | **0.920** |
| Built-up | 0.626 | 0.770 |
| Bare/Sparse | 0.092 | 0.168 |
| Wetland | 0.005 | 0.010 |

**Model-variant comparison**

| Model | Test OA | Test mIoU |
|---|---|---|
| U-Net **VV+VH** | **0.757** | **0.347** |
| U-Net VV-only | 0.684 | 0.303 |

Adding the VH (cross-pol) channel improves every major class (Trees +0.14 IoU,
Cropland +0.08, Built-up +0.02), confirming dual-polarization is worth it.

**Reading the numbers:** OA 0.76 from SAR alone is solid (radar is much harder than
optical). The lower mIoU is driven by three classes — Water (SAR confusion with
smooth/dark surfaces) and Bare/Wetland (0.4% of data each). The generalization gap
(val mIoU 0.545 → test 0.347) reflects the held-out, unseen test regions.

*Figures saved by train.py: `training_curves.png`, `confusion_matrix.png`, `class_distribution.png`.*